In [ ]:
# CELL 1 - mount + installs  (runs in ~1 min)
from google.colab import drive
drive.mount('/content/drive')
import sys, os
os.system(f"{sys.executable} -m pip -q install hnswlib sentence-transformers rapidfuzz")
print("ready")


In [ ]:
# ============================================================
#  SECTION A - WHICH INDEX DID THE 22-23 AUG RE-RUN ACTUALLY USE?
# ============================================================
import os, pickle, numpy as np, hnswlib
from sentence_transformers import SentenceTransformer

ROOT = "/content/drive/MyDrive"
BASE_ID = "Omartificial-Intelligence-Space/GATE-AraBert-v1"

# top-1 similarity recorded in the 23 Aug phase3_guardrail_results.json
RECORDED = {
 "ما فوائد الصبر في القرآن":   0.6228314638137817,
 "من هو النبي المعروف بالصبر": 0.5121996998786926,
 "ماذا يقول القرآن عن الرحمة": 0.6183292865753174,
 "التوبة والاستغفار":          0.7106841802597046,
 "العدل في الإسلام":           0.4078161120414734,
 "الشكر لله":                  0.7442700862884521,
 "الخوف من الله":              0.6203955411911011,
 "الايمان بالغيب":             0.6870181560516357,
 "الصدق في القول":             0.6918022632598877,
 "بر الوالدين":                0.5654799938201904,
}

def load_idx(d):
    ep = os.path.join(d, "entries.pkl"); hp = os.path.join(d, "verses.hnsw")
    if not (os.path.exists(ep) and os.path.exists(hp)):
        return None, None
    ent = pickle.load(open(ep, "rb"))
    ix = hnswlib.Index(space="cosine", dim=768)
    ix.load_index(hp)
    ix.set_ef(max(64, 10))
    return ix, ent

def probe(model, ix, ent, k=5):
    out = {}
    for q in RECORDED:
        emb = model.encode([q], convert_to_numpy=True, normalize_embeddings=True)
        lab, dist = ix.knn_query(emb, k=k)
        sims = (1.0 - dist[0]).tolist()
        out[q] = (sims, [ent[i].get("verse_key") for i in lab[0]],
                        [ent[i].get("source_type") for i in lab[0]])
    return out

def verdict(name, res):
    print(f"\n--- {name} ---")
    diffs = []
    for q, exp in RECORDED.items():
        sims, vks, sts = res[q]
        d = abs(sims[0] - exp)
        diffs.append(d)
        mark = "MATCH" if d < 1e-5 else ("close" if d < 1e-2 else "     ")
        print(f"  {mark} got {sims[0]:.10f}  recorded {exp:.10f}  diff {d:.2e}")
        print(f"        top5 {vks}")
        print(f"        types {sts}")
    n = sum(1 for d in diffs if d < 1e-5)
    print(f"\n  EXACT MATCHES: {n}/{len(diffs)}   max diff {max(diffs):.2e}")
    return n

print("=" * 70)
print("loading base model from HuggingFace ...")
base = SentenceTransformer(BASE_ID)
print("ok, dim =", base.get_sentence_embedding_dimension())

CAND = [
  ("BASE model + verses-only index (the recommended fix)",
   base, f"{ROOT}/Phase1_Project/index_rebuild_base_verses_only/index"),
]

results = {}
for name, mdl, d in CAND:
    ix, ent = load_idx(d)
    if ix is None:
        print(f"\n--- {name} ---\n  index not found at {d}")
        continue
    print(f"\n[{name}] index has {len(ent):,} entries")
    results[name] = verdict(name, probe(mdl, ix, ent))

if not results or max(results.values()) < len(RECORDED):
    print("\n" + "=" * 70)
    print("base+verses-only did NOT reproduce the numbers - trying the old setups")
    print("=" * 70)
    import shutil
    V2 = f"{ROOT}/Phase3_Project/guardrail_output_v2/b5_real_finetuned_v2"
    V1 = f"{ROOT}/Phase1_Project/MemberB_B4_B6_output/b5_real_finetuned"
    for label, mpath, ipath in [
        ("v2 fine-tuned + index_v2 (verses+tafsir)", V2,
         f"{ROOT}/Phase3_Project/guardrail_output_v2/index_v2"),
        ("v1 fine-tuned + MemberB index (verses+tafsir)", V1,
         f"{ROOT}/Phase1_Project/MemberB_B4_B6_output/index"),
        ("BASE model + index_v2 (verses+tafsir)", None,
         f"{ROOT}/Phase3_Project/guardrail_output_v2/index_v2"),
    ]:
        try:
            if mpath is None:
                mdl = base
            else:
                if not os.path.exists(mpath):
                    print(f"\n--- {label} ---\n  model missing: {mpath}"); continue
                local = "/content/" + os.path.basename(mpath)
                if not os.path.exists(local):
                    print(f"\ncopying {os.path.basename(mpath)} from Drive ...")
                    shutil.copytree(mpath, local,
                        ignore=shutil.ignore_patterns("checkpoint-*", "*.pt", "*.pth"))
                mdl = SentenceTransformer(local)
            ix, ent = load_idx(ipath)
            if ix is None:
                print(f"\n--- {label} ---\n  index not found at {ipath}"); continue
            print(f"\n[{label}] index has {len(ent):,} entries")
            verdict(label, probe(mdl, ix, ent))
        except Exception as e:
            print(f"\n--- {label} ---\n  FAILED: {type(e).__name__}: {e}")

print("\n" + "=" * 70)
print("SECTION A DONE")
print("=" * 70)


In [ ]:
# ============================================================
#  SECTION B - DOES THE NEW F1/F2 CALIBRATION ACTUALLY WORK?
# ============================================================
import os, sys, json, re, shutil, statistics as st

ROOT = "/content/drive/MyDrive"
SRC  = f"{ROOT}/Phase3_Project/guardrail_output/src"
WORK = "/content/g_src"

shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(SRC, WORK)
sys.path.insert(0, WORK)
print("copied", len(os.listdir(WORK)), "files to", WORK)

try:
    import rapidfuzz
except ImportError:
    os.system(f"{sys.executable} -m pip -q install rapidfuzz")

import importlib
f1 = f2 = None
try:
    f1 = importlib.import_module("f1_calibrate_guardrail")
    f2 = importlib.import_module("f2_build_calibration_dataset")
    print("imported f1 and f2 OK")
except Exception as e:
    print("COULD NOT IMPORT f1/f2:", type(e).__name__, e)

QRCD = None
for c in [f"{ROOT}/Phase4_Project/data/qrcd_flat.json",
          f"{ROOT}/Phase2_Project/Roma_output/data/qrcd_flat.json"]:
    if os.path.exists(c):
        QRCD = c; break
print("qrcd:", QRCD)

# ---------------------------------------------------------------- B1
print("\n" + "#" * 70)
print("# B1 - CALIBRATE ON QRCD DISTRACTOR-SWAPS (what f1+f2 are designed to do)")
print("#" * 70)
cal_tol = None
if f1 and f2 and QRCD:
    try:
        recs = f2.load_qrcd(QRCD)
        examples = f2.build_calibration_set(recs, n_pairs=40, seed=42)
        cal_tol, cal_acc, scored = f1.calibrate_max_acceptable_mismatch(examples)
        g = [s["mismatch_score"] for s in scored if s["label"] == "grounded"]
        h = [s["mismatch_score"] for s in scored if s["label"] == "hallucinated"]
        print(f"\n  grounded    mismatch: mean {st.mean(g):.3f}  min {min(g):.3f}  max {max(g):.3f}")
        print(f"  hallucinated mismatch: mean {st.mean(h):.3f}  min {min(h):.3f}  max {max(h):.3f}")
        print(f"  overlap between the two classes: "
              f"{'NONE - trivially separable' if max(g) < min(h) else 'yes, they overlap'}")
    except Exception as e:
        print("  FAILED:", type(e).__name__, e)
else:
    print("  skipped (missing f1/f2/qrcd)")

# ---------------------------------------------------------------- B2
print("\n" + "#" * 70)
print("# B2 - DOES THAT TOLERANCE TRANSFER TO THE 10 REAL RESPONSES?")
print("#" * 70)
RES = f"{ROOT}/Phase3_Project/guardrail_output/phase3_guardrail_results.json"
if f1 and os.path.exists(RES):
    real = json.load(open(RES, encoding="utf-8"))
    rows = []
    for d in real:
        ctx = d.get("context") or []
        inctx = {c.get("verse_key") for c in ctx if isinstance(c, dict)}
        cited = set(re.findall(r"\[(\d+:\d+)\]", str(d.get("final_response", ""))))
        if not cited:
            label = "no_citations"
        elif cited <= inctx:
            label = "grounded"
        else:
            label = "hallucinated"
        ex = {"query": d.get("query", "?"), "context": ctx,
              "response_text": d.get("final_response", ""), "label": label}
        try:
            m = f1.compute_mismatch_for_example(ex)
        except Exception as e:
            m = None
            print("   mismatch failed for", ex["query"][:30], type(e).__name__, e)
        rows.append({"q": ex["query"], "label": label, "recomputed": m,
                     "recorded": d.get("final_mismatch_score"),
                     "verified": d.get("verified"),
                     "n_cited": len(cited), "n_ungrounded": len(cited - inctx)})

    print("\n  query                          label         recomputed  recorded   diff")
    for r in rows:
        rc, rd = r["recomputed"], r["recorded"]
        diff = f"{abs(rc-rd):.4f}" if (rc is not None and rd is not None) else "  -  "
        rcs = f"{rc:.4f}" if rc is not None else " fail "
        print(f"  {r['q'][:28]:<30s} {r['label']:<13s} {rcs:>10s}  {rd:.4f}  {diff}")

    ok = [r for r in rows if r["recomputed"] is not None and r["recorded"] is not None]
    if ok:
        mx = max(abs(r["recomputed"] - r["recorded"]) for r in ok)
        print(f"\n  reproduction check: max diff {mx:.2e} "
              f"({'reproduces the saved run' if mx < 1e-6 else 'DOES NOT reproduce'})")

    ev = [r for r in rows if r["label"] in ("grounded", "hallucinated") and r["recomputed"] is not None]
    def score(tol):
        c = sum(1 for r in ev if (r["recomputed"] <= tol) == (r["label"] == "grounded"))
        return c / len(ev) if ev else 0.0
    print(f"\n  on the {len(ev)} real responses with citations:")
    print(f"    tolerance 0.50 (what the run used)      -> accuracy {score(0.5):.1%}")
    if cal_tol is not None:
        print(f"    tolerance {cal_tol:.2f} (calibrated on QRCD)      -> accuracy {score(cal_tol):.1%}")
    bt = max([x/100 for x in range(101)], key=score)
    print(f"    tolerance {bt:.2f} (best possible on these 10) -> accuracy {score(bt):.1%}")
    ng = sum(1 for r in ev if r["label"] == "grounded")
    print(f"    always-say-hallucinated baseline        -> accuracy {1-ng/len(ev):.1%}" if ev else "")
else:
    print("  skipped")

print("\n" + "=" * 70)
print("SECTION B DONE")
print("=" * 70)
